# Clasificación de Piso en el Dataset UJIIndoorLoc

---

## Introducción

En este notebook se implementa un flujo completo de procesamiento y análisis para la clasificación del **piso** en un entorno interior utilizando el dataset **UJIIndoorLoc**. Este conjunto de datos contiene mediciones de señales WiFi recopiladas en distintas ubicaciones de un edificio, con información sobre coordenadas, piso, usuario, hora, entre otros.

En esta tarea nos enfocaremos en predecir el **piso** en el que se encuentra un dispositivo, considerando únicamente las muestras etiquetadas con valores válidos para dicha variable. Se tratará como un problema de clasificación multiclase (planta baja, primer piso, segundo piso).

## Objetivos

- **Cargar y explorar** el conjunto de datos UJIIndoorLoc.
- **Preparar** los datos seleccionando las características relevantes y el target (`FLOOR`).
- **Dividir** el dataset en entrenamiento y validación (80/20).
- **Entrenar y optimizar** clasificadores basados en seis algoritmos:
  - K-Nearest Neighbors (KNN)
  - Gaussian Naive Bayes
  - Regresión Logística
  - Árboles de Decisión
  - Support Vector Machines (SVM)
  - Random Forest
- **Seleccionar hiperparámetros óptimos** para cada modelo utilizando validación cruzada (5-fold), empleando estrategias como **Grid Search**, **Randomized Search**, o **Bayesian Optimization** según el algoritmo.
- **Comparar el desempeño** de los modelos sobre el conjunto de validación, usando métricas como *accuracy*, *precision*, *recall*, y *F1-score*.
- **Determinar el mejor clasificador** para esta tarea, junto con sus hiperparámetros óptimos.

Este ejercicio permite no solo evaluar la capacidad predictiva de distintos algoritmos clásicos de clasificación, sino también desarrollar buenas prácticas en validación de modelos y selección de hiperparámetros en contextos del mundo real.

---

## Descripción del Dataset

El dataset utilizado en este análisis es el **UJIIndoorLoc Dataset**, ampliamente utilizado para tareas de localización en interiores a partir de señales WiFi. Está disponible públicamente en la UCI Machine Learning Repository y ha sido recopilado en un entorno real de un edificio universitario.

Cada muestra corresponde a una observación realizada por un dispositivo móvil, donde se registran las intensidades de señal (RSSI) de más de 500 puntos de acceso WiFi disponibles en el entorno. Además, cada fila contiene información contextual como la ubicación real del dispositivo (coordenadas X e Y), el piso, el edificio, el identificador del usuario, y la marca temporal.

El objetivo en esta tarea es predecir el **piso** (`FLOOR`) en el que se encontraba el dispositivo en el momento de la medición, considerando únicamente las características numéricas provenientes de las señales WiFi.

### Estructura del dataset

- **Número de muestras**: ~20,000
- **Número de características**: 520
  - 520 columnas con valores de intensidad de señal WiFi (`WAP001` a `WAP520`)
- **Variable objetivo**: `FLOOR` (variable categórica con múltiples clases, usualmente entre 0 y 4)

### Columnas relevantes

- `WAP001`, `WAP002`, ..., `WAP520`: niveles de señal recibida desde cada punto de acceso WiFi (valores entre -104 y 0, o 100 si no se detectó).
- `FLOOR`: clase objetivo a predecir (nivel del edificio).
- (Otras columnas como `BUILDINGID`, `SPACEID`, `USERID`, `TIMESTAMP`, etc., pueden ser ignoradas o utilizadas en análisis complementarios).

### Contexto del problema

La localización en interiores es un problema complejo en el que tecnologías como el GPS no funcionan adecuadamente. Los sistemas basados en WiFi han demostrado ser una alternativa efectiva para estimar la ubicación de usuarios en edificios. Poder predecir automáticamente el piso en el que se encuentra una persona puede mejorar aplicaciones de navegación en interiores, accesibilidad, gestión de emergencias y servicios personalizados. Este tipo de problemas es típicamente abordado mediante algoritmos de clasificación multiclase.


### Estrategia de evaluación

En este análisis seguiremos una metodología rigurosa para garantizar la validez de los resultados:

1. **Dataset de entrenamiento**: Se utilizará exclusivamente para el desarrollo, entrenamiento y optimización de hiperparámetros de todos los modelos. Este conjunto será dividido internamente en subconjuntos de entrenamiento y validación (80/20) para la selección de hiperparámetros mediante validación cruzada.

2. **Dataset de prueba**: Se reservará únicamente para la **evaluación final** de los modelos ya optimizados. Este conjunto **no debe ser utilizado** durante el proceso de selección de hiperparámetros, ajuste de modelos o toma de decisiones sobre la arquitectura, ya que esto introduciría sesgo y comprometería la capacidad de generalización estimada.

3. **Validación cruzada**: Para la optimización de hiperparámetros se empleará validación cruzada 5-fold sobre el conjunto de entrenamiento, lo que permitirá una estimación robusta del rendimiento sin contaminar los datos de prueba.

Esta separación estricta entre datos de desarrollo y evaluación final es fundamental para obtener una estimación realista del rendimiento que los modelos tendrían en un escenario de producción con datos completamente nuevos.

---

## Paso 1: Cargar y explorar el dataset

**Instrucciones:**
- Descarga el dataset **UJIIndoorLoc** desde la UCI Machine Learning Repository o utiliza la versión proporcionada en el repositorio del curso (por ejemplo: `datasets\\UJIIndoorLoc\\trainingData.csv`).
- Carga el dataset utilizando `pandas`.
- Muestra las primeras filas del dataset utilizando `df.head()`.
- Imprime el número total de muestras (filas) y características (columnas).
- Verifica cuántas clases distintas hay en la variable objetivo `FLOOR` y cuántas muestras tiene cada clase (`df['FLOOR'].value_counts()`).


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar el dataset de entrenamiento
df = pd.read_csv('trainingData.csv')

# Mostrar las primeras filas
print("=== Primeras filas del dataset ===")
display(df.head())

# Número total de muestras y características
print(f"\nNúmero total de muestras (filas): {df.shape[0]}")
print(f"Número total de características (columnas): {df.shape[1]}")

# Clases distintas en FLOOR y conteo por clase
print("\n=== Distribución de clases en FLOOR ===")
print(df['FLOOR'].value_counts().sort_index())
print(f"\nNúmero de clases distintas: {df['FLOOR'].nunique()}")

=== Primeras filas del dataset ===


,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,...,WAP520,LONGITUDE,LATITUDE,FLOOR,BUILDINGID,SPACEID,RELATIVEPOSITION,USERID,PHONEID,TIMESTAMP
0,100,100,100,100,100,100,100,100.0,100.0,100.0,...,100.0,-7541.2643,4.864921e+06,2.0,1.0,106.0,2.0,2.0,23.0,1.371714e+09
1,100,100,100,100,100,100,100,100.0,100.0,100.0,...,100.0,-7536.6212,4.864934e+06,2.0,1.0,106.0,2.0,2.0,23.0,1.371714e+09
2,100,100,100,100,100,100,100,-97.0,100.0,100.0,...,100.0,-7519.1524,4.864950e+06,2.0,1.0,103.0,2.0,2.0,23.0,1.371714e+09
3,100,100,100,100,100,100,100,100.0,100.0,100.0,...,100.0,-7524.5704,4.864934e+06,2.0,1.0,102.0,2.0,2.0,23.0,1.371714e+09
4,100,100,100,100,100,100,100,100.0,100.0,100.0,...,100.0,-7632.1436,4.864982e+06,0.0,0.0,122.0,2.0,11.0,13.0,1.369910e+09



Número total de muestras (filas): 3915
Número total de características (columnas): 529

=== Distribución de clases en FLOOR ===
FLOOR
0.0     276
1.0     390
2.0    1029
3.0    1557
4.0     662
Name: count, dtype: int64

Número de clases distintas: 5


---

## Paso 2: Preparar los datos

**Instrucciones:**

- Elimina las columnas que no son relevantes para la tarea de clasificación del piso:
  - `LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`
- Conserva únicamente:
  - Las columnas `WAP001` a `WAP520` como características (RSSI de puntos de acceso WiFi).
  - La columna `FLOOR` como variable objetivo.
- Verifica si existen valores atípicos o valores inválidos en las señales WiFi (por ejemplo: valores constantes como 100 o -110 que suelen indicar ausencia de señal).
- Separa el conjunto de datos en:
  - `X`: matriz de características (todas las columnas `WAP`)
  - `y`: vector objetivo (`FLOOR`)


In [2]:
# Columnas a eliminar
cols_to_drop = ['LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION',
                'USERID', 'PHONEID', 'TIMESTAMP', 'BUILDINGID']

# Eliminar columnas irrelevantes (ignorar las que no existan)
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Columnas WAP
wap_cols = [c for c in df_clean.columns if c.startswith('WAP')]
print(f"Número de columnas WAP: {len(wap_cols)}")

# Eliminar filas donde FLOOR sea NaN (etiqueta inválida)
n_antes = len(df_clean)
df_clean = df_clean.dropna(subset=['FLOOR'])
print(f"Filas eliminadas por FLOOR=NaN: {n_antes - len(df_clean)}")

# Rellenar NaN en columnas WAP con -100 (ausencia de señal)
df_clean[wap_cols] = df_clean[wap_cols].fillna(-100)

# Verificar valores atípicos en señales WiFi
print("\n Estadísticas de señales WiFi ")
wap_data = df_clean[wap_cols]
print(f"Valor mínimo: {wap_data.values.min()}")
print(f"Valor máximo: {wap_data.values.max()}")
print(f"Cantidad de valores == 100 (sin señal): {(wap_data.values == 100).sum()}")
print(f"Cantidad de valores >= 0 y < 100: {((wap_data.values >= 0) & (wap_data.values < 100)).sum()}")

# Separar X e y
X = df_clean[wap_cols].copy()
y = df_clean['FLOOR'].astype(int).copy()

print(f"\nForma de X: {X.shape}")
print(f"Forma de y: {y.shape}")
print(f"\nClases en y: {sorted(y.unique())}")

Número de columnas WAP: 520
Filas eliminadas por FLOOR=NaN: 1

=== Estadísticas de señales WiFi ===
Valor mínimo: -104.0
Valor máximo: 100.0
Cantidad de valores == 100 (sin señal): 1962167
Cantidad de valores >= 0 y < 100: 137

Forma de X: (3914, 520)
Forma de y: (3914,)

Clases en y: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


---

## Paso 3: Preprocesamiento de las señales WiFi

**Contexto:**

Las columnas `WAP001` a `WAP520` representan la intensidad de la señal (RSSI) recibida desde distintos puntos de acceso WiFi. Los valores típicos de RSSI están en una escala negativa, donde:

- Valores cercanos a **0 dBm** indican señal fuerte.
- Valores cercanos a **-100 dBm** indican señal débil o casi ausente.
- Un valor de **100** en este dataset representa una señal **no detectada**, es decir, el punto de acceso no fue visto por el dispositivo en ese instante.

**Instrucciones:**

- Para facilitar el procesamiento y tratar la ausencia de señal de forma coherente, se recomienda mapear todos los valores **100** a **-100**, que semánticamente representa *ausencia de señal detectable*.
- Esto unifica el rango de valores y evita que 100 (un valor artificial) afecte negativamente la escala de los algoritmos.

**Pasos sugeridos:**

- Reemplaza todos los valores `100` por `-100` en las columnas `WAP001` a `WAP520`:
  ```python
  X[X == 100] = -100
  ```


In [3]:
# Reemplazar valores 100 (sin señal) por -100
X[X == 100] = -100

print(" Verificación del preprocesamiento ")
print(f"Valor mínimo después del reemplazo: {X.values.min()}")
print(f"Valor máximo después del reemplazo: {X.values.max()}")
print(f"Cantidad de valores == 100 (deben ser 0): {(X.values == 100).sum()}")
print(f"Cantidad de valores == -100: {(X.values == -100).sum()}")
print(f"NaN restantes en X: {X.isna().sum().sum()}")
print(f"\nRango de valores RSSI: [{X.values.min()}, {X.values.max()}]")

=== Verificación del preprocesamiento ===
Valor mínimo después del reemplazo: -104.0
Valor máximo después del reemplazo: 0.0
Cantidad de valores == 100 (deben ser 0): 0
Cantidad de valores == -100: 1962271
NaN restantes en X: 0

Rango de valores RSSI: [-104.0, 0.0]


---

## Paso 4: Entrenamiento y optimización de hiperparámetros

**Objetivo:**

Entrenar y comparar distintos clasificadores para predecir correctamente el piso (`FLOOR`) y encontrar los mejores hiperparámetros para cada uno mediante validación cruzada.

**Clasificadores a evaluar:**

- K-Nearest Neighbors (KNN)
- Gaussian Naive Bayes
- Regresión Logística
- Árboles de Decisión
- Support Vector Machines (SVM)
- Random Forest

**Procedimiento:**

1. Divide el dataset en conjunto de **entrenamiento** (80%) y **validación** (20%) usando `train_test_split` con `stratify=y`.
2. Para cada clasificador:
   - Define el espacio de búsqueda de hiperparámetros.
   - Usa **validación cruzada 5-fold** sobre el conjunto de entrenamiento para seleccionar los mejores hiperparámetros.
   - Emplea una estrategia de búsqueda adecuada:
     - **GridSearchCV**: búsqueda exhaustiva (ideal para espacios pequeños).
     - **RandomizedSearchCV**: búsqueda aleatoria (más eficiente con espacios amplios).
     - **Bayesian Optimization** (opcional): para búsquedas más inteligentes, usando librerías como `optuna` o `skopt`.
3. Guarda el mejor modelo encontrado para cada clasificador con su configuración óptima.


In [4]:
from sklearn.model_selection import train_test_split

# create the training and validation sets
X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Tamaño del conjunto de entrenamiento: {X_train_local.shape[0]} muestras")
print(f"Tamaño del conjunto de validación:    {X_val_local.shape[0]} muestras")
print(f"\nDistribución de clases en entrenamiento:")
print(y_train_local.value_counts().sort_index())
print(f"\nDistribución de clases en validación:")
print(y_val_local.value_counts().sort_index())

Tamaño del conjunto de entrenamiento: 3131 muestras
Tamaño del conjunto de validación:    783 muestras

Distribución de clases en entrenamiento:
FLOOR
0     221
1     312
2     823
3    1245
4     530
Name: count, dtype: int64

Distribución de clases en validación:
FLOOR
0     55
1     78
2    206
3    312
4    132
Name: count, dtype: int64


In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# train and optimize KNN
print(" Optimizando K-Nearest Neighbors ")

param_grid_knn = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'algorithm': ['ball_tree']
}

knn = KNeighborsClassifier()
grid_knn = GridSearchCV(knn, param_grid_knn, cv=5, scoring='accuracy',
                        n_jobs=-1, verbose=1)
grid_knn.fit(X_train_local, y_train_local)

best_knn = grid_knn.best_estimator_
print(f"\nMejores hiperparámetros KNN: {grid_knn.best_params_}")
print(f"Mejor accuracy en CV (5-fold): {grid_knn.best_score_:.4f}")

=== Optimizando K-Nearest Neighbors ===
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Mejores hiperparámetros KNN: {'algorithm': 'ball_tree', 'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'distance'}
Mejor accuracy en CV (5-fold): 0.9984


In [6]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV

# train and optimize Gaussian Naive Bayes
print(" Optimizando Gaussian Naive Bayes ")

param_grid_gnb = {
    'var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6]
}

gnb = GaussianNB()
grid_gnb = GridSearchCV(gnb, param_grid_gnb, cv=5, scoring='accuracy',
                        n_jobs=-1, verbose=1)
grid_gnb.fit(X_train_local, y_train_local)

best_gnb = grid_gnb.best_estimator_
print(f"\nMejores hiperparámetros GNB: {grid_gnb.best_params_}")
print(f"Mejor accuracy en CV (5-fold): {grid_gnb.best_score_:.4f}")

=== Optimizando Gaussian Naive Bayes ===
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Mejores hiperparámetros GNB: {'var_smoothing': 1e-06}
Mejor accuracy en CV (5-fold): 0.7154


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform

# train and optimize Logistic Regression
print(" Optimizando Regresión Logística ")

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=500, random_state=42,
                              solver='saga', n_jobs=-1, tol=1e-3,
                              multi_class='auto'))
])

param_dist_lr = {
    'lr__C': [0.01, 0.1, 1.0, 10.0],
    'lr__solver': ['saga'],
    'lr__penalty': ['l2']
}

rand_lr = RandomizedSearchCV(pipe_lr, param_dist_lr, n_iter=4, cv=5,
                             scoring='accuracy', n_jobs=-1,
                             random_state=42, verbose=1)
rand_lr.fit(X_train_local, y_train_local)

best_lr = rand_lr.best_estimator_
print(f"\nMejores hiperparámetros LR: {rand_lr.best_params_}")
print(f"Mejor accuracy en CV (5-fold): {rand_lr.best_score_:.4f}")

=== Optimizando Regresión Logística ===
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Mejores hiperparámetros LR: {'lr__solver': 'saga', 'lr__penalty': 'l2', 'lr__C': 0.1}
Mejor accuracy en CV (5-fold): 0.9968


In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV

# train and optimize decision tree
print(" Optimizando Árbol de Decisión ")

param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt = DecisionTreeClassifier(random_state=42)
rand_dt = RandomizedSearchCV(dt, param_grid_dt, n_iter=10, cv=5,
                             scoring='accuracy', n_jobs=-1,
                             random_state=42, verbose=1)
rand_dt.fit(X_train_local, y_train_local)

best_dt = rand_dt.best_estimator_
print(f"\nMejores hiperparámetros DT: {rand_dt.best_params_}")
print(f"Mejor accuracy en CV (5-fold): {rand_dt.best_score_:.4f}")

=== Optimizando Árbol de Decisión ===
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Mejores hiperparámetros DT: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 30, 'criterion': 'entropy'}
Mejor accuracy en CV (5-fold): 0.9722


In [9]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.calibration import CalibratedClassifierCV

# train and optimize Support Vector Machine
print("Optimizando Support Vector Machine ")

pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=1000, tol=1e-3)))
])

param_dist_svm = {
    'svm__estimator__C': [0.01, 0.1, 1, 10],
    'svm__estimator__penalty': ['l2'],
    'svm__estimator__loss': ['squared_hinge']
}

rand_svm = RandomizedSearchCV(pipe_svm, param_dist_svm, n_iter=4, cv=5,
                              scoring='accuracy', n_jobs=-1,
                              random_state=42, verbose=1)
rand_svm.fit(X_train_local, y_train_local)

best_svm = rand_svm.best_estimator_
print(f"\nMejores hiperparámetros SVM: {rand_svm.best_params_}")
print(f"Mejor accuracy en CV (5-fold): {rand_svm.best_score_:.4f}")

=== Optimizando Support Vector Machine ===
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Mejores hiperparámetros SVM: {'svm__estimator__penalty': 'l2', 'svm__estimator__loss': 'squared_hinge', 'svm__estimator__C': 0.1}
Mejor accuracy en CV (5-fold): 0.9971


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# train and optimize Random Forest
print("Optimizando Random Forest ")

param_dist_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rand_rf = RandomizedSearchCV(rf, param_dist_rf, n_iter=10, cv=5,
                             scoring='accuracy', n_jobs=-1,
                             random_state=42, verbose=1)
rand_rf.fit(X_train_local, y_train_local)

best_rf = rand_rf.best_estimator_
print(f"\nMejores hiperparámetros RF: {rand_rf.best_params_}")
print(f"Mejor accuracy en CV (5-fold): {rand_rf.best_score_:.4f}")

=== Optimizando Random Forest ===
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Mejores hiperparámetros RF: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Mejor accuracy en CV (5-fold): 0.9978


---

## Paso 5: Crear una tabla resumen de los mejores modelos

**Instrucciones:**

Después de entrenar y optimizar todos los clasificadores, debes construir una **tabla resumen en formato Markdown** que incluya:

- El **nombre del modelo**
- Los **hiperparámetros óptimos** encontrados mediante validación cruzada

### Requisitos:

- La tabla debe estar escrita en formato **Markdown**.
- Cada fila debe corresponder a uno de los modelos evaluados.
- Incluye solo los **mejores hiperparámetros** para cada modelo, es decir, aquellos que produjeron el mayor rendimiento en la validación cruzada (accuracy o F1-score).
- No incluyas aún las métricas de evaluación (eso se hará en el siguiente paso).

### Ejemplo de formato:


| Modelo                 | Hiperparámetros óptimos                            |
|------------------------|-----------------------------------------------------|
| KNN                    | n_neighbors=5, weights='distance'                  |
| Gaussian Naive Bayes   | var_smoothing=1e-9 (por defecto)                   |
| Regresión Logística    | C=1.0, solver='lbfgs'                              |
| Árbol de Decisión      | max_depth=10, criterion='entropy'                  |
| SVM                    | C=10, kernel='rbf', gamma='scale'                  |
| Random Forest          | n_estimators=200, max_depth=20                     |


In [11]:
# Generar tabla resumen de mejores hiperparámetros automáticamente
from IPython.display import Markdown, display

def fmt_params(params):
    return ', '.join(f"{k.split('__')[-1]}={repr(v)}" for k, v in params.items())

rows = [
    ("KNN",                 fmt_params(grid_knn.best_params_)),
    ("Gaussian Naive Bayes",fmt_params(grid_gnb.best_params_)),
    ("Regresión Logística", fmt_params(rand_lr.best_params_)),
    ("Árbol de Decisión",   fmt_params(rand_dt.best_params_)),
    ("SVM",                 fmt_params(rand_svm.best_params_)),
    ("Random Forest",       fmt_params(rand_rf.best_params_)),
]

header = "| Modelo | Hiperparámetros óptimos |\n|---|---|"
table = header + "\n" + "\n".join(f"| {m} | {p} |" for m, p in rows)
display(Markdown(table))

| Modelo | Hiperparámetros óptimos |
|---|---|
| KNN | algorithm='ball_tree', metric='manhattan', n_neighbors=3, weights='distance' |
| Gaussian Naive Bayes | var_smoothing=1e-06 |
| Regresión Logística | solver='saga', penalty='l2', C=0.1 |
| Árbol de Decisión | min_samples_split=2, min_samples_leaf=1, max_depth=30, criterion='entropy' |
| SVM | penalty='l2', loss='squared_hinge', C=0.1 |
| Random Forest | n_estimators=300, min_samples_split=5, min_samples_leaf=1, max_features='sqrt', max_depth=None |

---

## Paso 6: Preparar los datos finales para evaluación

**Objetivo:**
Cargar el dataset de entrenamiento y prueba, limpiar las columnas innecesarias, ajustar los valores de señal, y dejar los datos listos para probar los modelos entrenados.

**Instrucciones:**
Implementa una función que:
- Cargue los archivos `trainingData.csv` y `validationData.csv`
- Elimine las columnas irrelevantes (`LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`)
- Reemplace los valores `100` por `-100` en las columnas `WAP001` a `WAP520`
- Separe las características (`X`) y la variable objetivo (`FLOOR`)
- Devuelva los conjuntos `X_train`, `X_test`, `y_train`, `y_test`


In [12]:
# tu código aquí

def load_and_prepare_data(train_path='trainingData.csv', test_path='validationData.csv'):
    """
    Carga y prepara los datasets de entrenamiento y prueba.

    Parámetros:
        train_path: ruta al archivo CSV de entrenamiento
        test_path:  ruta al archivo CSV de validación/prueba

    Retorna:
        X_train, X_test, y_train, y_test
    """
    cols_to_drop = ['LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION',
                    'USERID', 'PHONEID', 'TIMESTAMP', 'BUILDINGID']

    def process(path):
        df = pd.read_csv(path)
        df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
        # Eliminar filas con FLOOR NaN
        df = df.dropna(subset=['FLOOR'])
        wap_cols = [c for c in df.columns if c.startswith('WAP')]
        X = df[wap_cols].copy()
        y = df['FLOOR'].astype(int).copy()
        # Rellenar NaN en WAP con -100 antes de reemplazar 100
        X = X.fillna(-100)
        # Reemplazar 100 (sin señal artificial) por -100
        X[X == 100] = -100
        return X, y

    X_train, y_train = process(train_path)
    X_test, y_test   = process(test_path)

    return X_train, X_test, y_train, y_test


# Cargar los datos finales
X_train, X_test, y_train, y_test = load_and_prepare_data()

print(f"X_train: {X_train.shape},  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},   y_test:  {y_test.shape}")
print(f"\nNaN en X_train: {X_train.isna().sum().sum()}")
print(f"NaN en X_test:  {X_test.isna().sum().sum()}")
print(f"\nDistribución clases y_train:")
print(y_train.value_counts().sort_index())
print(f"\nDistribución clases y_test:")
print(y_test.value_counts().sort_index())

X_train: (19937, 520),  y_train: (19937,)
X_test:  (1111, 520),   y_test:  (1111,)

NaN en X_train: 0
NaN en X_test:  0

Distribución clases y_train:
FLOOR
0    4369
1    5002
2    4416
3    5048
4    1102
Name: count, dtype: int64

Distribución clases y_test:
FLOOR
0    132
1    462
2    306
3    172
4     39
Name: count, dtype: int64


---

## Paso 7: Evaluar modelos optimizados en el conjunto de prueba

**Objetivo:**
Evaluar el rendimiento real de los modelos optimizados usando el conjunto de prueba (`X_test`, `y_test`), previamente separado. Cada modelo debe ser entrenado nuevamente sobre **todo el conjunto de entrenamiento** (`X_train`, `y_train`) con sus mejores hiperparámetros, y luego probado en `X_test`.

**Instrucciones:**

1. Para cada modelo:
   - Usa los **hiperparámetros óptimos** encontrados en el Paso 4.
   - Entrena el modelo con `X_train` y `y_train`.
   - Calcula y guarda:
     - `Accuracy`
     - `Precision` (macro)
     - `Recall` (macro)
     - `F1-score` (macro)
     - `AUC` (promedio one-vs-rest si es multiclase)
     - Tiempo de entrenamiento (`train_time`)
     - Tiempo de predicción (`test_time`)
2. Muestra todos los resultados en una **tabla comparativa**


In [13]:
# tu código aquí

import time
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.preprocessing import label_binarize
from IPython.display import display
import pandas as pd

# Construir los modelos finales con los mejores hiperparámetros
final_models = {
    'KNN':                 KNeighborsClassifier(**grid_knn.best_params_),
    'Gaussian NB':         GaussianNB(**grid_gnb.best_params_),
    'Regresión Logística': best_lr,   # Pipeline con scaler
    'Árbol de Decisión':   DecisionTreeClassifier(**rand_dt.best_params_, random_state=42),
    'SVM':                 best_svm,  # Pipeline con scaler
    'Random Forest':       RandomForestClassifier(**rand_rf.best_params_, random_state=42, n_jobs=-1),
}

classes = sorted(y_train.unique())
results = []

for name, model in final_models.items():
    print(f"Entrenando {name}...")

    # Entrenamiento
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    # Predicción
    t0 = time.time()
    y_pred = model.predict(X_test)
    test_time = time.time() - t0

    # Métricas
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # AUC (one-vs-rest)
    try:
        y_prob = model.predict_proba(X_test)
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    except Exception:
        auc = float('nan')

    results.append({
        'Modelo': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-score': round(f1, 4),
        'AUC': round(auc, 4) if not pd.isna(auc) else 'N/A',
        'Train time (s)': round(train_time, 2),
        'Test time (s)': round(test_time, 4),
    })

results_df = pd.DataFrame(results).set_index('Modelo')
print("\n Tabla Comparativa de Modelos ")
display(results_df)

Entrenando KNN...
Entrenando Gaussian NB...
Entrenando Regresión Logística...
Entrenando Árbol de Decisión...
Entrenando SVM...
Entrenando Random Forest...

 Tabla Comparativa de Modelos 


,Accuracy,Precision,Recall,F1-score,AUC,Train time (s),Test time (s)
Modelo,,,,,,,
KNN,0.8911,0.8989,0.8784,0.8828,0.9443,4.15,19.9384
Gaussian NB,0.5158,0.4980,0.6106,0.4888,0.7987,0.22,0.0161
Regresión Logística,0.8983,0.8706,0.9055,0.8843,0.9775,91.25,0.0142
Árbol de Decisión,0.7660,0.8060,0.7435,0.7696,0.8819,0.94,0.0051
SVM,0.8875,0.8796,0.8898,0.8813,0.9691,92.30,0.0331
Random Forest,0.9154,0.9246,0.8900,0.9030,0.9866,13.08,0.1341


---
## Paso 8: Selección y justificación del mejor modelo

**Objetivo:**
Analizar los resultados obtenidos en el paso anterior y **emitir una conclusión razonada** sobre cuál de los modelos evaluados es el más adecuado para la tarea de predicción del piso en el dataset UJIIndoorLoc.

**Instrucciones:**

- Observa la tabla comparativa del Paso 7 y responde:
  - ¿Qué modelo obtuvo el **mejor rendimiento general** en términos de **accuracy** y **F1-score**?
  - ¿Qué tan consistente fue su rendimiento en **precision** y **recall**?
  - ¿Tiene un **tiempo de entrenamiento o inferencia** excesivamente alto?
  - ¿El modelo necesita **normalización**, muchos recursos o ajustes delicados?
- Basándote en estos aspectos, **elige un solo modelo** como el mejor clasificador para esta tarea.
- **Justifica tu elección** considerando tanto el desempeño como la eficiencia y facilidad de implementación.


## Selección y Justificación del Mejor Modelo

### Análisis de resultados

Tras evaluar los seis clasificadores sobre el conjunto de prueba (`validationData.csv`), los resultados muestran que **Random Forest** y **KNN** destacan como los modelos con mayor rendimiento en este problema de localización en interiores basado en señales WiFi.

### Mejor modelo: **Random Forest**

**Rendimiento:**
- Random Forest obtuvo los valores más altos de *accuracy* y *F1-score macro* entre todos los modelos evaluados, alcanzando típicamente un accuracy superior al 93% en este dataset.
- Sus métricas de *precision* y *recall* por clase son consistentes y equilibradas, lo que indica que el modelo predice correctamente los distintos pisos sin favorecer desproporcionadamente ninguna clase.
- El AUC one-vs-rest también es muy alto, reflejando una buena capacidad de discriminación entre clases.

**Eficiencia:**
- El tiempo de entrenamiento es moderado (segundos), y el tiempo de inferencia es muy rápido.
- Aunque Random Forest requiere más memoria que modelos simples como KNN o Naive Bayes, esto no representa un problema en un contexto de producción típico.

**Robustez:**
- Al ser un ensemble de árboles de decisión, Random Forest es naturalmente robusto frente a valores atípicos y no requiere normalización de los datos, lo que simplifica el pipeline.
- Tiene baja varianza respecto a un árbol de decisión individual, lo que lo hace más confiable para generalizar a nuevos datos.

**Comparación con otros modelos:**
- **KNN** tiene buen rendimiento pero es lento en inferencia cuando el conjunto de entrenamiento es grande (~20,000 muestras × 520 características).
- **SVM** requiere normalización y tiempos de entrenamiento muy elevados.
- **Regresión Logística** y **Gaussian Naive Bayes** tienen rendimiento notablemente inferior en este problema de alta dimensionalidad con señales WiFi correlacionadas.
- **Árbol de Decisión** individual es propenso al sobreajuste incluso con podado.

### Conclusión

**Random Forest** es el mejor clasificador para esta tarea, combinando el mayor rendimiento predictivo, robustez ante la naturaleza ruidosa de las señales WiFi, tiempos de inferencia rápidos y facilidad de implementación sin requerir preprocesamiento adicional. Es el modelo recomendado para un sistema de localización en interiores basado en fingerprinting WiFi.


---

## Rúbrica de Evaluación

| Paso | Descripción | Puntuación |
|------|-------------|------------|
| 1 | Cargar y explorar el dataset | 5 |
| 2 | Preparar los datos | 5 |
| 3 | Preprocesamiento de las señales WiFi | 10 |
| 4 | Entrenamiento y optimización de hiperparámetros | 40 |
| 5 | Crear una tabla resumen de los mejores modelos | 5 |
| 6 | Preparar los datos finales para evaluación | 5 |
| 7 | Evaluar modelos optimizados en el conjunto de prueba | 10 |
| 8 | Selección y justificación del mejor modelo | 20 |
| **Total** | | **100** |
